In [1]:
from analytical import *
from configuration_sampler import *
from engine import *
from trajectory import *
from electronic import *
import numpy as np
from matplotlib import pyplot as plt


In [2]:
bound_A = -0.8
bound_B = 0.8

def in_stateA(q, s):
    return q<bound_A and s == 0

def in_stateB(q, s):
    return q>bound_B and s == 0

In [ ]:
num_samples = 5000
mass = 1836.15
temp = [1263, 2105, 2526, 3158, 4210]
temp = temp[::-1]
epsilon = 0.05
Vc = 0.2 * epsilon
SHO_H = DoubleHarmonic(k=2*epsilon, x0=1.0, vc=Vc, mass=mass)
dt = 5
max_timestep = 3000
E_a = 0.8 * epsilon

ratio_lst = []
TP_ensemble_T = []

for temperature in temp:
    print('Tempearture:', temperature)
    q_inits, p_inits, S_inits = generate_mash_initial_conditions(num_samples=num_samples, mass=mass,
                                                                temp=temperature, epsilon=epsilon,
                                                                Vc=Vc)
    coeff_inits = spin_to_electronic_coefficients(S_inits)
    ensemble = []
    extracted_paths = []
    print('Initial conditions generated')

    skip_count = 0
    for q_init, p_init, coeff_init in zip(q_inits, p_inits, coeff_inits):
        E_tot = get_energy(q_init, p_init, m=mass, epsilon=epsilon, Vc=Vc)
        if E_tot < E_a:
            skip_count += 1
            continue
        v_init = p_init/mass
        S_z_check = sz_from_coeff(coeff_init)
        if S_z_check >= 0:
            raise ValueError('Sampling of incorrect electronic variables')
        act_state_init = 0
        C_init = SHO_H.eigvecs(q_init)
        initial_snapshot = Snapshot(positions=q_init,
                                    velocities=v_init,
                                    coefficients=coeff_init,
                                    active_state=act_state_init,
                                    gauge=C_init,
                                    mass=mass,
                                    is_grid=True)
        
        MASH_engine = MASHEngine(model=SHO_H,
                         dt=dt,
                         initial_snapshot=initial_snapshot)
        trajectory = MASH_engine.propagate(max_timestep)
        ensemble.append(trajectory)
    
    print('raw ensemble generated')
    
    for traj in ensemble:
        current_basin = None
        last_basin_idx = None
        for i, snap in enumerate(traj):
            q = snap.positions
            s = snap.active_state

            if in_stateA(q, s):
                if current_basin == 'B':
                    transition_path = traj[last_basin_idx : i + 1]
                    extracted_paths.append(transition_path)
                current_basin = 'A'
                last_basin_idx = i
            
            elif in_stateB(q, s):
                if current_basin == 'A':
                    transition_path = traj[last_basin_idx : i + 1]
                    extracted_paths.append(transition_path)
                current_basin = 'B'
                last_basin_idx = i
    
    if len(extracted_paths) == 0:
        print('Not a single transition path found!')
        ratio_lst.append('inf')
        continue

    total_step_in_ensemble = max_timestep * num_samples
    ratio = total_step_in_ensemble/len(extracted_paths)
    ratio_lst.append(ratio)

    TP_ensemble_T.append(extracted_paths)
    print(skip_count)
    


Tempearture: 4210
MCMC Acceptance Rate for q: 64.16%
Initial conditions generated
raw ensemble generated
4798
Tempearture: 3158
MCMC Acceptance Rate for q: 58.87%
Initial conditions generated
raw ensemble generated
4918
Tempearture: 2526
MCMC Acceptance Rate for q: 54.58%
Initial conditions generated
raw ensemble generated
4977
Tempearture: 2105
MCMC Acceptance Rate for q: 51.49%
Initial conditions generated
raw ensemble generated
4988
Tempearture: 1263
MCMC Acceptance Rate for q: 43.41%
Initial conditions generated
raw ensemble generated
Not a single transition path found!


In [5]:
print(ratio_lst)

[7751.937984496124, 17709.56316410862, 72115.38461538461, 122950.81967213115, 'inf']


In [7]:
num_samples = 5000
mass = 1836.15
epsilon = 0.05
Vc = 0.2 * epsilon
SHO_H = DoubleHarmonic(k=2*epsilon, x0=1.0, vc=Vc, mass=mass)
dt = 5
max_timestep = 3000
E_a = 0.8 * epsilon

test_temp = 12000
q_inits, p_inits, S_inits = generate_mash_initial_conditions(num_samples=num_samples, mass=mass,
                                                                temp=test_temp, epsilon=epsilon,
                                                                Vc=Vc)
coeff_inits = spin_to_electronic_coefficients(S_inits)

ensemble = []
extracted_paths = []
print('Initial conditions generated')

skip_count = 0
for q_init, p_init, coeff_init in zip(q_inits, p_inits, coeff_inits):
    E_tot = get_energy(q_init, p_init, m=mass, epsilon=epsilon, Vc=Vc)
    if E_tot < E_a:
        skip_count += 1
        continue
    v_init = p_init/mass
    S_z_check = sz_from_coeff(coeff_init)
    if S_z_check >= 0:
        raise ValueError('Sampling of incorrect electronic variables')
    act_state_init = 0
    C_init = SHO_H.eigvecs(q_init)
    initial_snapshot = Snapshot(positions=q_init,
                                velocities=v_init,
                                coefficients=coeff_init,
                                active_state=act_state_init,
                                gauge=C_init,
                                mass=mass,
                                is_grid=True)
    
    MASH_engine = MASHEngine(model=SHO_H,
                        dt=dt,
                        initial_snapshot=initial_snapshot)
    trajectory = MASH_engine.propagate(max_timestep)
    ensemble.append(trajectory)

print('raw ensemble generated')
for traj in ensemble:
    current_basin = None
    last_basin_idx = None
    for i, snap in enumerate(traj):
        q = snap.positions
        s = snap.active_state

        if in_stateA(q, s):
            if current_basin == 'B':
                transition_path = traj[last_basin_idx : i + 1]
                extracted_paths.append(transition_path)
            current_basin = 'A'
            last_basin_idx = i
        
        elif in_stateB(q, s):
            if current_basin == 'A':
                transition_path = traj[last_basin_idx : i + 1]
                extracted_paths.append(transition_path)
            current_basin = 'B'
            last_basin_idx = i


total_step_in_ensemble = max_timestep * num_samples
    
if len(extracted_paths) == 0:
        print('Not a single transition path found!')
        ratio_lst.append('inf')
else:
    ratio = total_step_in_ensemble/len(extracted_paths)

print(skip_count)

transition_path_ensemble = extracted_paths

mean_transition_time = np.mean([(len(transition_path_ensemble[i])-1)*dt for i in range(len(transition_path_ensemble))])

MCMC Acceptance Rate for q: 80.35%
Initial conditions generated
raw ensemble generated
3482


In [8]:
print(mean_transition_time)

353.3718150146868
